In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# =================================================================
# PASO 1: Cargar los Datos Ya Preprocesados
# =================================================================
df_limpio = pd.read_csv('data/partidos_entrenamiento.csv',parse_dates=['date'])
df_limpio

# =================================================================
# PASO 2: Definir las Características (X) y el Objetivo (y)
# =================================================================
# Incluimos de forma estricta los nuevos predictores que calcula tu script
features = [
    'goles_totales_marcados_local_10', 'goles_totales_concedidos_local_10', 'forma_puntos_local_10',
    'goles_totales_marcados_visitante_10', 'goles_totales_concedidos_visitante_10', 'forma_puntos_visitante_10',
    'dif_goles_marcados_10', 'dif_goles_concedidos_10', 'dif_forma_puntos_10',
    'puntos_fifa_local', 'puntos_fifa_visitante', 'dif_puntos_fifa',
    'ranking_pos_local', 'ranking_pos_visitante',
    'neutral', 'importancia_torneo'
]

X = df_limpio[features]
valid_idx = df_limpio['resultado'].notna()
X = X.loc[valid_idx]
y = df_limpio.loc[valid_idx, 'resultado'].astype(int)

# Separación aleatoria conservando el 20% para el examen de control
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# =================================================================
# PASO 3: Entrenar el Modelo con tus Hiperparámetros Optimizados
# =================================================================
modelo_mundial = HistGradientBoostingClassifier(
    max_iter=80,
    learning_rate=0.03,
    max_depth=6,
    class_weight='balanced',      # Clave para evitar el sesgo local y capturar empates
    random_state=99
)

modelo_mundial.fit(X_train, y_train)

# Evaluación de control obligatoria para ver las nuevas métricas
predicciones_test = modelo_mundial.predict(X_test)
print(f"Precisión General de Control: {accuracy_score(y_test, predicciones_test) * 100:.2f}%\n")
print("Reporte detallado de Control (1=Local, 0=Empate, 2=Visitante):")
print(classification_report(y_test, predicciones_test))

print("-" * 65)

# =================================================================
# PASO 4: FUNCIÓN NATIVA DE MONTECARLO PARA EL MUNDIAL 2026
# =================================================================
def simular_partido_montecarlo(datos_partido_X, n_simulaciones=10000):
    """
    Recibe los predictores de un partido y simula su resultado 10,000 veces
    usando las probabilidades reales distribuidas por el Gradiente Boosting.
    """
    # 1. Extraer las probabilidades asignadas por el modelo entrenado
    # predict_proba devuelve una matriz con: [P(Empate), P(Gana Local), P(Gana Visitante)]
    probabilidades = modelo_mundial.predict_proba(datos_partido_X)[0]
    
    p_empate = probabilidades[0]
    p_local = probabilidades[1]
    p_visitante = probabilidades[2]
    
    # 2. Definir las etiquetas de salida en consonancia con tu CSV
    opciones_resultado = [0, 1, 2] # 0=Empate, 1=Local, 2=Visitante
    
    # 3. Lanzar la tómbola aleatoria de Montecarlo basada en pesos probabilísticos
    universos_simulados = np.random.choice(
        opciones_resultado, 
        size=n_simulaciones, 
        p=[p_empate, p_local, p_visitante]
    )
    
    # 4. Consolidar los resultados del muestreo estadístico
    locales_ganados = np.sum(universos_simulados == 1)
    empates_totales = np.sum(universos_simulados == 0)
    visitantes_ganados = np.sum(universos_simulados == 2)
    
    # 5. Imprimir el diagnóstico del universo paralelo
    print(f"Probabilidades Base del Modelo -> Local: {p_local*100:.2f}% | Empate: {p_empate*100:.2f}% | Visitante: {p_visitante*100:.2f}%")
    print(f"Resultado tras {n_simulaciones} simulaciones de Montecarlo:")
    print(f" -> Victorias del Local: {locales_ganados} veces ({locales_ganados/n_simulaciones*100:.2f}%)")
    print(f" -> Empates calculados: {empates_totales} veces ({empates_totales/n_simulaciones*100:.2f}%)")
    print(f" -> Victorias del Visitante: {visitantes_ganados} veces ({visitantes_ganados/n_simulaciones*100:.2f}%)\n")
    
    return universos_simulados


print("Simulando 1000 Montecarlo para TODOS los partidos...")

# 1. Obtenemos las matrices de probabilidades para TODOS los partidos del tirón
# Esto nos devuelve una lista de probabilidades por cada fila de X_test
todas_las_probabilidades = modelo_mundial.predict_proba(X_test)

lista_resultados_montecarlo = []

# 2. Recorremos cada partido y ejecutamos sus 10,000 simulaciones independientes
for probs in todas_las_probabilidades:
    p_empate = probs[0]
    p_local = probs[1]
    p_visitante = probs[2]
    
    # Lanzamos las 10,000 tiradas de ruleta para este partido específico
    simulaciones_partido = np.random.choice(
        [0, 1, 2], 
        size=10000, 
        p=[p_empate, p_local, p_visitante]
    )
    
    # Calculamos los porcentajes finales obtenidos en Montecarlo
    pct_local = (np.sum(simulaciones_partido == 1) / 10000) * 100
    pct_empate = (np.sum(simulaciones_partido == 0) / 10000) * 100
    pct_visitante = (np.sum(simulaciones_partido == 2) / 10000) * 100
    
    # Guardamos las métricas de este partido
    lista_resultados_montecarlo.append({
        'prob_montecarlo_local_%': round(pct_local, 2),
        'prob_montecarlo_empate_%': round(pct_empate, 2),
        'prob_montecarlo_visitante_%': round(pct_visitante, 2)
    })

# 3. Convertimos los resultados en una tabla ordenada de Pandas
df_probabilidades_totales = pd.DataFrame(lista_resultados_montecarlo)
df_probabilidades_totales['resultado final'] = y_test.reset_index(drop=True)

# Mostramos las primeras 5 filas para verificar que calculó todos los partidos
print("\n¡Simulación completada con éxito para todos los partidos!")
print(df_probabilidades_totales.head())

Precisión General de Control: 54.40%

Reporte detallado de Control (1=Local, 0=Empate, 2=Visitante):
              precision    recall  f1-score   support

           0       0.30      0.35      0.32      1397
           1       0.70      0.61      0.65      2967
           2       0.54      0.58      0.56      1739

    accuracy                           0.54      6103
   macro avg       0.51      0.52      0.51      6103
weighted avg       0.56      0.54      0.55      6103

-----------------------------------------------------------------
Simulando 1000 Montecarlo para TODOS los partidos...

¡Simulación completada con éxito para todos los partidos!
   prob_montecarlo_local_%  prob_montecarlo_empate_%  \
0                    42.75                     35.62   
1                    28.25                     32.34   
2                     5.19                     10.65   
3                    26.72                     48.18   
4                    20.75                     38.00   

   

In [11]:
from sklearn.metrics import log_loss

# 1. Obtenemos las probabilidades puras del modelo sobre el examen (X_test)
probabilidades_test = modelo_mundial.predict_proba(X_test)

# 2. Calculamos la pérdida logarítmica comparando con los resultados reales (y_test)
error_log_loss = log_loss(y_test, probabilidades_test)

print(f"Desviación Log Loss del modelo: {error_log_loss:.4f}")

Desviación Log Loss del modelo: 0.9471


In [12]:
from sklearn.metrics import brier_score_loss

# 1. Creamos una lista real de: ¿Fue empate? (1 si sí, 0 si no)
empates_reales = (y_test == 0).astype(int)

# 2. Extraemos la probabilidad de empate que asignó tu simulación (columna índice 0)
probabilidades_empate = probabilidades_test[:, 0]

# 3. Calculamos la desviación de Brier
desviacion_brier_empate = brier_score_loss(empates_reales, probabilidades_empate)

print(f"Desviación Brier para los Empates: {desviacion_brier_empate:.4f}")

Desviación Brier para los Empates: 0.1813


In [13]:
import pandas as pd
import numpy as np

# 1. Cargamos el archivo limpio que generó tu script de preprocesamiento
df_entrenamiento = pd.read_csv('data/partidos_entrenamiento.csv', parse_dates=['date'])

# 2. Creamos un diccionario vacío para guardar el estado de cada país
diccionario_maestro = {}

# 3. Agrupamos todos los equipos que existen en la base de datos (tanto locales como visitantes)
todas_las_selecciones = set(df_entrenamiento['home_team'].unique()).union(set(df_entrenamiento['away_team'].unique()))

for seleccion in todas_las_selecciones:
    # Buscamos los partidos donde participó esta selección
    partidos_equipo = df_entrenamiento[(df_entrenamiento['home_team'] == seleccion) | (df_entrenamiento['away_team'] == seleccion)]
    
    if not partidos_equipo.empty:
        # Tomamos el último partido de su historia (el más reciente en la línea de tiempo)
        ultimo_partido = partidos_equipo.sort_values('date').iloc[-1]
        
        # Extraemos sus métricas dependiendo de si en ese último partido jugó como local o visitante
        if ultimo_partido['home_team'] == seleccion:
            goles_recientes = ultimo_partido['media_goles_recientes_local']
            goles_concedidos = ultimo_partido['media_goles_concedidos_local']
            forma_reciente = ultimo_partido['forma_reciente_local']
            peso_cam = ultimo_partido['peso_camiseta_local']
        else:
            goles_recientes = ultimo_partido['media_goles_recientes_visitante']
            goles_concedidos = ultimo_partido['media_goles_concedidos_visitante']
            forma_reciente = ultimo_partido['forma_reciente_visitante']
            peso_cam = ultimo_partido['peso_camiseta_visitante']
            
        # Guardamos su "ficha técnica" en el diccionario maestro
        diccionario_maestro[seleccion] = {
            'goles_recientes': goles_recientes,
            'goles_concedidos': goles_concedidos,
            'forma_reciente': forma_reciente,
            'peso_camiseta': peso_cam
        }

print(f"¡Diccionario Maestro creado con éxito! Se ha guardado el estado actual de {len(diccionario_maestro)} selecciones mundiales.")


KeyError: 'media_goles_recientes_visitante'

In [ ]:
def predecir_encuentro_mundial(local, visitante, n_simulaciones=1000):
    # 1. Verificar si ambas selecciones existen en el diccionario maestro
    if local not in diccionario_maestro or visitante not in diccionario_maestro:
        return f"Error: Uno o ambos equipos ('{local}' / '{visitante}') no tienen suficiente historial en el CSV."
        
    # 2. Extraer los datos guardados de cada país de forma automatizada
    loc = diccionario_maestro[local]
    vis = diccionario_maestro[visitante]
    
    # 3. Construir la fila de 11 predictores calculando las diferencias (Locales menos Visitantes)
    datos_partido = {
        'importancia_torneo': 4.0, # Nivel de máxima importancia (Mundial)
        'ventaja_localia': 0,      # 0 porque es campo neutral en el Mundial
        'media_goles_recientes_local': loc['goles_recientes'],
        'media_goles_recientes_visitante': vis['goles_recientes'],
        'forma_reciente_local': loc['forma_reciente'],
        'forma_reciente_visitante': vis['forma_reciente'],
        
        # Calculamos los predictores diferenciales automáticos
        'dif_goles_recientes': loc['goles_recientes'] - vis['goles_recientes'],
        'dif_forma_reciente': loc['forma_reciente'] - vis['forma_reciente'],
        'dif_goles_concedidos': loc['goles_concedidos'] - vis['goles_concedidos'],
        'dif_peso_camiseta': loc['peso_camiseta'] - vis['peso_camiseta']
    }
    
    # Convertimos en el DataFrame estructurado que exige tu HistGradientBoosting
    X_partido = pd.DataFrame([datos_partido])
    
    # 4. Extraer las probabilidades asignadas por tu modelo entrenado
    probabilidades = modelo_mundial.predict_proba(X_partido[features])[0]
    
    p_empate = probabilidades[0]
    p_local = probabilidades[1]
    p_visitante = probabilidades[2]
    
    # 5. Ejecutar la ruleta de Montecarlo en base a las probabilidades de la IA
    simulaciones = np.random.choice([0, 1, 2], size=n_simulaciones, p=[p_empate, p_local, p_visitante])
    
    # 6. Calcular los porcentajes consolidados del torneo simulado
    pct_local = (np.sum(simulaciones == 1) / n_simulaciones) * 100
    pct_empate = (np.sum(simulaciones == 0) / n_simulaciones) * 100
    pct_visitante = (np.sum(simulaciones == 2) / n_simulaciones) * 100
    
    # Mostrar resultados limpios y formateados en pantalla
    print(f"=== SIMULACIÓN MONTECARLO: {local} vs {visitante} ===")
    print(f"Probabilidad de Victoria de {local}: {pct_local:.2f}%")
    print(f"Probabilidad de Empate: {pct_empate:.2f}%")
    print(f"Probabilidad de Victoria de {visitante}: {pct_visitante:.2f}%\n")
    
    return {'local': pct_local, 'empate': pct_empate, 'visitante': pct_visitante}

# -----------------------------------------------------------------
# ¡PRUEBA TU NUEVO MOTOR PREDICTIVO AUTOMÁTICO!
# -----------------------------------------------------------------
# Escribe cualquier partido que aparezca en tu fixture (Respeta las mayúsculas del CSV original)
_ = predecir_encuentro_mundial('Spain', 'Uruguay')

